# 주9: 수익률

**202315163 김소희**

삼성전자의 최근 1개월 일별 주가를 수집하여 5일 이동평균, 단순수익률, 로그수익률을 계산합니다. 첫 거래일과 마지막 거래일의 종가로 구한 기간 수익률을 일별 수익률 컬럼의 누적값과 비교합니다.


## 1. 주가 데이터 수집

분석 종목은 삼성전자(005930)이며, 2026년 11월 1일을 기준으로 최근 1개월인 2026년 10월 1일부터 11월 1일까지의 일별 주가를 조회합니다. 주말과 휴장일을 제외한 실제 자료는 **10월 1일부터 10월 30일까지 총 20거래일**입니다.

FinanceDataReader로 주가를 수집하고 시가(Open), 고가(High), 저가(Low), 종가(Close), 거래량(Volume)을 선택하여 데이터프레임을 구성합니다. 가격 단위는 원이며, 수익률은 종가를 기준으로 계산합니다.


In [1]:
import FinanceDataReader as fdr
import pandas as pd
import numpy as np

start = '2026-10-01'
end = '2026-11-01'

df = fdr.DataReader('005930', start, end)
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

print('종목: 삼성전자 (005930)')
print(f'조회 기간: {start} ~ {end}')
print(f'실제 자료 기간: {df.index[0]:%Y-%m-%d} ~ {df.index[-1]:%Y-%m-%d}')
print(f'거래일 수: {len(df)}일')


종목: 삼성전자 (005930)
조회 기간: 2026-10-01 ~ 2026-11-01
실제 자료 기간: 2026-10-01 ~ 2026-10-30
거래일 수: 20일


## 2. 이동평균 및 일별 수익률 계산

거래일 $t$의 종가를 $P_t$라고 할 때 각 컬럼의 계산식은 다음과 같습니다.

| 컬럼 | 의미 | 계산식 |
| --- | --- | --- |
| MA5 | 5일 이동평균 | $(P_t+P_{t-1}+P_{t-2}+P_{t-3}+P_{t-4})/5$ |
| returnsDaily | 일별 단순수익률 | $r_t=P_t/P_{t-1}-1$ |
| returnsLogDaily | 일별 로그수익률 | $\ell_t=\ln(P_t/P_{t-1})$ |

5일 이동평균은 현재 거래일을 포함한 최근 5개 거래일 종가의 평균입니다. 처음 4개 행은 관측값이 부족하여 이동평균이 `NaN`입니다. 첫 행의 두 수익률도 이전 종가가 없어 `NaN`으로 표시됩니다.

수익률 컬럼은 소수 단위입니다. 예를 들어 단순수익률 `0.01`은 `1%`를 의미합니다.


In [2]:
df['MA5'] = df['Close'].rolling(window=5).mean()
df['returnsDaily'] = df['Close'].pct_change()
df['returnsLogDaily'] = np.log(df['Close'] / df['Close'].shift(1))

df


,Open,High,Low,Close,Volume,MA5,returnsDaily,returnsLogDaily
Date,,,,,,,,
2026-10-01,266500,276000,264500,274500,13741073,NaN,NaN,NaN
2026-10-02,273500,277000,271500,276000,11501250,NaN,0.005464,0.005450
2026-10-06,278500,279000,270000,273000,12913819,NaN,-0.010870,-0.010929
2026-10-07,269500,279500,268000,269000,16395504,NaN,-0.014652,-0.014760
2026-10-08,269500,270000,262000,263000,21259056,271100.0,-0.022305,-0.022557
2026-10-12,260000,262500,249000,251000,19771955,266400.0,-0.045627,-0.046701
2026-10-13,241500,244000,228000,234000,26790913,258000.0,-0.067729,-0.070132
2026-10-14,233500,241500,230000,234500,19009052,250300.0,0.002137,0.002134
2026-10-15,216500,221500,204500,207500,34330565,238000.0,-0.115139,-0.122324


## 3. 기간 수익률 계산 및 비교

조회된 첫 거래일의 종가를 $P_0$, 마지막 거래일의 종가를 $P_T$라고 할 때 기간 단순수익률과 로그수익률은 다음과 같습니다.

$$
R=\frac{P_T}{P_0}-1
$$

$$
L=\ln\left(\frac{P_T}{P_0}\right)
$$

일별 컬럼에서는 $(1+r_t)$를 누적하여 곱한 후 1을 빼면 기간 단순수익률을 구할 수 있습니다. 로그수익률은 일별 값을 누적하여 더하면 됩니다. 각각 `cumprod()`와 `cumsum()`으로 계산한 마지막 값을 종가로 직접 구한 값과 비교합니다.


In [3]:
first_close = df['Close'].iloc[0]
last_close = df['Close'].iloc[-1]

# 첫 종가와 마지막 종가로 계산
simple_return = last_close / first_close - 1
log_return = np.log(last_close / first_close)

# 일별 수익률 컬럼을 누적하여 계산
simple_from_column = ((1 + df['returnsDaily']).cumprod() - 1).iloc[-1]
log_from_column = df['returnsLogDaily'].cumsum().iloc[-1]

comparison = pd.DataFrame({
    '종가로 계산': [simple_return, log_return],
    '컬럼으로 계산': [simple_from_column, log_from_column]
}, index=['단순수익률', '로그수익률'])

print(f'첫 종가: {first_close:,.0f}원')
print(f'마지막 종가: {last_close:,.0f}원')
print(f'기간 단순수익률: {simple_return:.6%}')
print(f'기간 로그수익률: {log_return:.6f}')

comparison


첫 종가: 274,500원
마지막 종가: 220,500원
기간 단순수익률: -19.672131%
기간 로그수익률: -0.219054


,종가로 계산,컬럼으로 계산
단순수익률,-0.196721,-0.196721
로그수익률,-0.219054,-0.219054


## 4. 결과 분석

삼성전자의 종가는 2026년 10월 1일 **274,500원**에서 10월 30일 **220,500원**으로 하락했습니다. 두 시점 사이의 단순수익률과 로그수익률은 각각 **-19.672131%** 및 **-0.219054**입니다. 종가로 직접 구한 두 수익률은 각각 일별 컬럼의 누적값과 일치합니다.

일별 단순수익률을 그대로 합하면 **-19.322717%** 로 기간 단순수익률과 다릅니다. 매일의 수익률은 서로 다른 전일 종가를 기준으로 계산되므로 기간 전체의 단순수익률은 복리 누적으로 계산해야 합니다.

반면 로그수익률은 일별 값을 합산할 수 있습니다. 누적 로그수익률에 지수함수를 적용하고 1을 빼면 기간 단순수익률 **-19.672131%** 로 환산됩니다. 분석 결과는 종가의 변화에 따른 가격수익률이며, 배당, 수수료, 세금은 별도로 반영하지 않았습니다.


In [4]:
print(f"일별 단순수익률의 합: {df['returnsDaily'].sum():.6%}")
print(f'일별 단순수익률의 복리 누적: {simple_from_column:.6%}')
print(f'일별 로그수익률의 누적합: {log_from_column:.6f}')
print(f'로그수익률을 단순수익률로 환산: {np.exp(log_from_column) - 1:.6%}')


일별 단순수익률의 합: -19.322717%
일별 단순수익률의 복리 누적: -19.672131%
일별 로그수익률의 누적합: -0.219054
로그수익률을 단순수익률로 환산: -19.672131%


## 데이터 출처

[네이버 증권 삼성전자](https://stock.naver.com/domestic/stock/005930/price). FinanceDataReader 0.9.202로 수집했으며, 수집일은 2026년 11월 1일입니다.
